### 分值转移
**题目描述**
给定一个长度为n的数组{a1, a2,..., an}。你可以进行如下操作至多一次（也可以不进行任何操作）：

选择一个下标1 ≤ i＜ n 和任意整数 x ，将区间a1, a2,..., ai 中的每个元素减少 x，同
时将区间 ai+1，ai+2，⋯，an 中的每个元素增加 x。

请你最小化操作结束后数组中的逆序对个数，并输出这个最小值。

【名词解释】
逆序对：对在序列中，若存在两个下标 p, q 满足 `p <q`且 `ap > aq`，则称 (p, q) 为一个逆序对。

**输入描述**
每个测试文件均包含多组测试数据。第一行输入一个整数 (1≤ t ≤ 10^4) 表示数据组数，每组测试数据描述如下：
1. 第一行输入一个正整数 n (1≤ n ≤ 10^5) 表示数组的长度。
2. 第二行输入 n 个正整数 a1, a2, ..., an (1≤ ai ≤ n) 表示数组 a。
3. 保证单个测试文件的 n 之和不超过 2*10^5。
**输出描述**
对于每一组测试数据，新起一行，输出一个整数，表示最多可以进行的操作次数。

样例输入：
```text
2
5
2 1 5 3 4
4
4 3 2 1
```
输出：
```text
1
2
```

#### 1.树状数组 快速查找逆序对
**题目操作**为：
- 选择一个分割点 i （1 < i < n）和一个整数 x，使得前缀 [1, i]减去 x，后缀[i + 1, n]加上 x。
- 等价于：对于任意一对下标 (p,q) 其中 p ≤ i< q,
  - 原值：a[p], a[q] 
  - 新值：a[p] - x, a[q] + x
  - 如果 x 足够大，那么 i 前缀的数会变得非常小，i 后缀的数会变非常大。此时 **跨 i 的逆序对 全部消失**。
  - i 前后两侧的逆序对 仍然存在。
**问题转化**：
- 一次操作，目标是找到一个分割点 i，使得跨越该点的逆序对数量最大。
- 余下 逆序对，等价于 i 两侧子序列中的逆序对。 即找到 i，使得 逆序对 min(L[1, i] + R[i + 1, n])。
**算法步骤**：
- 依次遍历数组中每个位置 i，计算 i 两侧子序列的逆序对数量。
- 取出 i 两侧子序列的逆序对数量中的最小值。
- 使用  树状数组（Fenwick Tree），时间复度 O(nlogn)。


In [ ]:
import sys
input = sys.stdin.readline

# 树状数组模板，用于高效求逆序对
class FenwickTree:
    def __init__(self, size):
        self.size = size
        self.tree = [0] * (size + 2) #下标从1kais
    
    # 单点更新
    def update(self, index, delta = 1):
        while index <= self.size:
            self.tree[index] += delta
            index += index & -index # lowbit 不懂

    # 前缀求和
    def query(self, index):
        res = 0
        while index > 0:
            res += self.tree[index]
            index -= index & -index # lowbit 不懂
        return res
    
def solve():
    t = int(input())  # 测试用例数
    for _ in range(t):
        n = int(input())
        a = list(map(int, input().split()))
        if n == 1:  # 长度为1，无逆序对
            print(0)
            continue
    
        max_val = n # 题目规定a[i]≤n
        # 1. 计算前缀逆序对 pre[i]：前i个元素的逆序对
        pre = [0]*(n+1)
        ft = FenwickTree(max_val) #初始化树状数组大小为n
        cnt = 0
        for i in range(n): # 依次遍历数组 对 i 前计算逆序对
            # 求比a[i]大的数的个数 = 已插入总数 - 比a[i]小的数的个数
            cnt += i - ft.query(a[i])
            pre[i] = cnt
            ft.update(a[i]) # 将a[i]插入树状数组

        # 2. 计算后缀逆序对 suf[i]：从i到n的逆序对
        suf = [0]*(n+2)
        ft = FenwickTree(max_val) # 重新初始化树状数组
        cnt = 0
        for i in range(n-1, -1, -1): # 逆序遍历数组 对 i 后计算逆序对
            cnt += ft.query(a[i]-1) # 求比a[i]小的数的个数
            suf[i + 1] = cnt
            ft.update(a[i]) # 将a[i]插入树状数组

        # 找最小值：原始逆序对 vs 所有分割点的结果
        total = pre[n]
        min_ans = total
        # 枚举分割点
        for i in range(1, n):
            current = pre[i] + suf[i + 1]
            if current < min_ans: # 找到最小值
                min_ans = current
        print(min_ans)

if __name__ == "__main__":
    solve()

#### 2. 尝试暴力双循环（不考虑 n 的数值范围）——可能超时
暴力循环用来理解题目逻辑要求，最直白。

**暴力求逆序对**：双重循环遍历所有 `p < q`，统计满足 `a[p]>a[q]` 的数量。

In [ ]:
import sys
input = sys.stdin.readline

def count_inversion(arr):
    cnt = 0
    n = len(arr)
    for p in range(n):
        for q in range(p+1, n): # 枚举所有的p和q，p<q
            if arr[p] > arr[q]:
                cnt += 1
    return cnt


def solve():
    t = int(input())  # 测试用例数
    for _ in range(t): # 依次处理每个测试用例
        n = int(input()) # 数组长度
        a = list(map(int, input().split())) # 数组元素
        if n == 1:  # 长度为1，无逆序对
            print(0)
            continue

        # 1. 原始数组的总逆序对（不操作的情况）
        total = count_inversion(a)
        min_ans = total # 初始化最小逆序对数为原始总数

        # 2. 枚举分割点，计算每个分割点的逆序对数
        for i in range(1, n):
            left = a[:i] # 左半部分
            right = a[i:] # 右半部分
            current = count_inversion(left) + count_inversion(right) # 当前分割点的逆序对数
            if current < min_ans: # 更新最小逆序对数
                min_ans = current

        print(min_ans) # 输出结果

if __name__ == "__main__":
    solve()
    

In [ ]:
# 测试样例
import sys
input = sys.stdin.readline

def count_inversion(arr):
    cnt = 0
    n = len(arr)
    for p in range(n):
        for q in range(p+1, n): # 枚举所有的p和q，p<q
            if arr[p] > arr[q]:
                cnt += 1
    return cnt

# 写死测试用例
data = ['2', '5', '3 1 2 5 4', '4', '4 3 2 1'] 
t = int(data[0])  # 测试用例数
for i in range(1, 2*t, 2):
    n = int(data[i]) # 数组长度
    a = list(map(int, data[i+1].split())) # 数组元素
    if n == 1:  # 长度为1，无逆序对
        print(0)
        continue

    # 1. 原始数组的总逆序对（不操作的情况）
    total = count_inversion(a)
    min_ans = total # 初始化最小逆序对数为原始总数

    # 2. 枚举分割点，计算每个分割点的逆序对数
    for i in range(1, n):
        left = a[:i] # 左半部分
        right = a[i:] # 右半部分
        current = count_inversion(left) + count_inversion(right) # 当前分割点的逆序对数
        if current < min_ans: # 更新最小逆序对数
            min_ans = current

    print(min_ans) # 输出结果

    

1
2
